# SAM-AI ARC-AGI-2 Frontier Solver (LB 33.89% SOTA Architecture)
### Developed by Parallax | Founder: Samrish B
**Core Technology:** Task-Time Training (TTT) with Unsloth FastLanguageModel + Turbo DFS Beam Search + Dynamic Multi-GPU Fair-Share Scheduling (rc_schedule.py).

- **Architecture:** Task-Time Fine-Tuning (TTT) on 128 task-specific augmentations per test task.
- **Decoding:** 12-token constrained Turbo Depth-First Search with 16 cross-validation inference views.
- **Accelerator:** 4x NVIDIA L4 GPUs via 	orch.multiprocessing.
- **Fault-Tolerance:** Full-queue scheduler, dynamic remaining-time fair-share allocator, resilient competition directory resolution.


In [ ]:
# Keep the original global 10-minute submission/write buffer.
import time
global_end_time = time.time() + 12 * 3600 - 600

In [ ]:
# Preserve the baseline environment workaround.
!pip uninstall -y tensorflow

In [ ]:
%%writefile arc_loader.py
import json
import numpy as np
from transformers import AutoTokenizer


def convert_grid_to_string(grid) -> str:
    text = ""
    for row in grid:
        for cell in row:
            text += str(int(cell))
        text += "\n"
    return text.strip()

def is_valid_solution(guess):
    return isinstance(guess, np.ndarray) and guess.ndim == 2 and all(0 < x <= 30 for x in guess.shape)

def shuffled(data_list):
    return np.random.permutation(data_list).tolist()

def permute_mod(a, descriptor, invert=False):
    permutation = [int(i) for i in descriptor if str(i).isdigit()]
    assert sorted(permutation)==list(range(10))
    a = np.asarray(a)
    if a.ndim==3:
        if not invert: permutation = np.argsort(permutation)
        a = a[..., permutation]
    else:
        assert a.ndim==2
        if invert: permutation = np.argsort(permutation)
        a = np.asarray(permutation)[a]
    return a

def permute_rnd_all_(query):
    permutation = np.random.permutation(10).tolist()
    return 'permute' + ''.join(map(str, permutation))


class QwenFormatter:

    def __init__(self, tokenizer: AutoTokenizer):
        self.tokenizer = tokenizer

    def fmt_query(self, query) -> str:
        grid_input = convert_grid_to_string(query[0]["input"])
        return "<|im_start|>user\n" + grid_input + "<|im_end|><|im_start|>assistant\n"

    def fmt_reply(self, reply) -> str:
        return convert_grid_to_string(reply[0]) + "<|im_end|>"

    def fmt_train(self, train, last_is_challenge=False) -> str:
        if last_is_challenge:
            test = train[-1]
            train = train[:-1]
        else:
            test = None
        text = ""
        for x in train:
            grid_input = convert_grid_to_string(x["input"])
            grid_output = convert_grid_to_string(x["output"])
            text += f"<|im_start|>user\n{grid_input}<|im_end|><|im_start|>assistant\n{grid_output}<|im_end|>"
        if test is not None:
            text += self.fmt_query([test]) + self.fmt_reply([test["output"]])
        return text

    def max_new_tokens(self):
        max_sized_reply = np.zeros([30, 30], dtype=int)
        tokens = self.tokenizer.encode(self.fmt_reply([max_sized_reply]))
        return len(tokens) + 1

    def convert_tokens_to_array(self, tokens, limit_rows=30):
        if len(tokens) < 2:
            return None
        text = self.tokenizer.decode(tokens[:-1])
        try:
            lines = text.strip().split("\n")
            by_rows = [row for row in [[int(x) for x in line if x.isdigit()] for line in lines] if len(row)]
            if len(by_rows) > limit_rows:
                by_rows = by_rows[:limit_rows]
            array = np.array(by_rows, dtype=int)
            if is_valid_solution(array):
                return array
        except:
            pass
        return None


class ArcDataset:

    @staticmethod
    def forward_mod(a, key, use_perm=True):
        if a is None: return a
        for op in key.split('.')[1:]:
            if   op=='rot90':              a = np.rot90(a)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=False) if use_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    @staticmethod
    def invert_mod(a, key, inv_perm=True):
        if a is None: return a
        for op in key.split('.')[1:][::-1]:
            if   op=='rot90':              a = np.rot90(a, k=3)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=True) if inv_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    def __init__(self, queries, replies={}, keys=None, is_orig=False):
        if keys is not None: keys = [k for k in keys if k is not None]
        self.queries = queries if keys is None else {k: queries[k] for k in keys}
        self.replies = replies if keys is None else {k: replies[k] for k in keys if k in replies}
        self.is_orig = is_orig
        self.keys = sorted(queries.keys()) if keys is None else keys
        self.transposed_dataset = None

    def change_keys(self, keys, keep_flags=False):
        flags = dict(is_orig=self.is_orig) if keep_flags else {}
        return self.__class__(queries=self.queries, replies=self.replies, keys=keys, **flags)

    @classmethod
    def from_file(cls, queries_file, keys=None):
        with open(queries_file) as f:
            queries = f.read()
        return cls(
            queries=json.loads(queries),
            is_orig=True,
            keys=keys,
        )

    def load_replies(self, replies_file):
        print(f"*** Load solutions from '{replies_file}'...")
        with open(replies_file) as f: replies = f.read()
        replies_parsed = json.loads(replies)
        self.replies = {k: replies_parsed[k] for k in self.keys}
        return self

    def split_multi_replies(self):
        key_indices = [(k, i) for k in self.keys for i in range(len(self.queries[k]['test']))]
        return self.__class__(
            keys=[f'{k}_{i}' for k, i in key_indices],
            queries={f'{k}_{i}': {'train': self.queries[k]['train'], 'test': [self.queries[k]['test'][i]]} for k, i in key_indices},
            replies={f'{k}_{i}': [self.replies[k][i]] for k, i in key_indices if k in self.replies},
        )

    def shuffled(self):
        return self.__class__(queries=self.queries, replies=self.replies, keys=shuffled(self.keys))

    def append(*datasets):
        return datasets[0].__class__(
            queries={k: v for d in datasets for k, v in d.queries.items()},
            replies={k: v for d in datasets for k, v in d.replies.items()},
            keys   =[k    for d in datasets for k    in d.keys           ],
        )

    def mod_single(self, mod_func, descriptor, i, keep_key, inputs_only):
        queries = {}
        replies = {}
        keys    = []
        for k0 in self.keys:
            desc = (('copy{i}' if mod_func is np.copy else mod_func.__name__) if descriptor is None else descriptor if isinstance(descriptor, str) else descriptor(self.queries[k0])).format(i=i)
            func = lambda a, d: np.asarray(mod_func(a) if descriptor is None else mod_func(a, d)).tolist()
            k1 = k0 if keep_key else f"{k0}.{'I' if inputs_only else ''}{desc}"
            keys.append(k1)
            queries[k1] = {m: [{t: (func(a, desc) if t=='input' or not inputs_only else a) for t, a in x.items()} for x in e] for m, e in self.queries[k0].items()}
            if k0 in self.replies:
                replies[k1] = [func(a, desc) for a in self.replies[k0]]
        ret = self.__class__(queries=queries, replies=replies, keys=keys)
        return ret

    def mod(self, mod_func, descriptor=None, n=1, stack=None, keep=False, keep_key=False, shuffle=False, join=True, inputs_only=False):
        assert not (keep and keep_key)
        cur = self
        ret = [cur.shuffled() if shuffle else cur] if keep else []
        if stack is None: stack = mod_func.__name__.startswith('rot')
        for i in range(n):
            cur = (cur if stack else self).mod_single(mod_func, descriptor, i=i, keep_key=keep_key, inputs_only=inputs_only)
            ret.append(cur.shuffled() if shuffle else cur)
        return self.__class__.append(*ret) if join else ret

    def get(self, key, formatter: QwenFormatter):
        train = formatter.fmt_train(self.queries[key]['train'])
        query = formatter.fmt_query(self.queries[key]['test'])
        reply = formatter.fmt_reply(self.replies[key]) if key in self.replies else ''
        text = train+query+reply if reply else formatter.fmt_train(self.queries[key]['train'], last_is_challenge=True)
        return dict(key=key, train=train, query=query, reply=reply, input=train+query, text=text)

    def as_list(self, formatter: QwenFormatter):
        return [self.get(key, formatter) for key in self.keys]

    def get_length(self, key, formatter: QwenFormatter, name, max_of_transposed=False):
        if formatter is None:
            if   name=='input': return sum(np.prod(np.shape(v)) for v3 in self.queries[key].values() for v2 in v3 for v in v2.values())
            elif name=='reply': return sum(np.prod(np.shape(v)) for v in self.replies[key])
            else: assert False
        else:
            datasets = [self]
            if max_of_transposed:
                if self.transposed_dataset is None: self.transposed_dataset = self.mod(np.transpose, keep=False, keep_key=True)
                datasets.append(self.transposed_dataset)
            return max(len(formatter.tokenizer.encode(ds.get(key, formatter=formatter)[name])) for ds in datasets)

    def cut_to_len(self, formatter, name, max_len, from_end=False):
        temp_ds = self.change_keys(self.keys)
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            reply = temp_ds.replies.get(key)
            while max_len<temp_ds.get_length(key, formatter=formatter, name=name):
                query = temp_ds.queries[key]
                if not key.split('.')[-1].startswith('ex'):
                    key = f"{key}.ex{''.join(map(str, range(len(query['train']))))}"
                key_split = key.split('.')
                assert key_split[-1].startswith('ex')
                key = '.'.join(key_split[:-1] + [f'ex{key_split[-1][2:-1] if from_end else key_split[-1][3:]}'])
                temp_ds.queries[key] = {k: ((v[:-1] if from_end else v[1:]) if k=='train' else v) for k, v in query.items()}
                if reply is not None:
                    temp_ds.replies[key] = reply
            new_keys.append(key)
            new_queries[key] = temp_ds.queries[key]
            if reply is not None: new_replies[key] = reply
        return self.__class__(keys=new_keys, queries=new_queries, replies=new_replies)
    
    def shuffle_ex(self, perm=None, keep_max=None):
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            n = len(self.queries[key]['train'])
            p = np.random.permutation(n) if perm is None else perm
            if keep_max is not None: p = p[:keep_max]
            new_key = f'{key}.ex' + ('-' if (p.max()>9) else '').join(map(str, p.tolist()))
            new_keys.append(new_key)
            new_queries[new_key] = {k: (np.array(v, dtype=object)[p].tolist() if k=='train' else v) for k, v in self.queries[key].items()}
            if key in self.replies: new_replies[new_key] = self.replies[key]
        return self.__class__(queries=new_queries, replies=new_replies, keys=new_keys)

    def augment(self, n=1, shfl_keys=False, seed=42):
        np.random.seed(seed)
        d = self
        d = d.mod(np.transpose, keep=True)
        d = d.mod(np.rot90, n=3, keep=True)
        d = d.mod(permute_mod, permute_rnd_all_, n=n, shuffle=shfl_keys, keep=False)
        d = d.shuffle_ex()
        return d

    def get_submission(self, results=None):
        assert self.is_orig==True, 'Must be run on original dataset.'
        submission = {k: [{f'attempt_{i+1}': [[0]] for i in range(2)} for _ in range(len(self.queries[k]['test']))] for k in self.keys}
        if results is not None: self.fill_submission(results, submission)
        return submission

    @staticmethod
    def fill_submission(results, submission):
        print(f'*** Generating submission for {len(results)} outputs...')
        for k, v in results.items():
            base_id, base_nr = k.split('_')
            target_dict = submission[base_id][int(base_nr)]
            for i, g in enumerate(v[:len(target_dict)]):
                target_dict[f'attempt_{i+1}'] = g.tolist()

    def validate_submission(self, submission):
        assert self.is_orig==True, 'Must be run on original dataset.'
        score = 0
        for k, v in self.replies.items():
            for i, r in enumerate(v):
                for attempt in ['attempt_1', 'attempt_2']:
                    if np.array_equal(r, submission[k][i][attempt]):
                        score += 1 / len(v)
                        break
        return score

In [ ]:
%%writefile arc_decoder.py
import os
import bz2
import pickle
import numpy as np

def hashable(guess):
    return tuple(map(tuple, guess))

def score_sum(guesses, getter):
    guess_list = list(guesses.values())
    scores = {}
    for g in guess_list:
        h = hashable(g["solution"])
        x = scores[h] = scores.get(h, [[], g["solution"]])
        x[0].append(g)
    scores = [(getter(sc), o) for sc, o in scores.values()]
    scores = sorted(scores, key=(lambda x: x[0]), reverse=True)
    ordered_outputs = [x[-1] for x in scores]
    return ordered_outputs

def getter_full_probmul_3(guesses, baseline=3):
    inf_score = np.sum([baseline-g["beam_score"] for g in guesses])
    aug_score = np.mean([np.sum([baseline-s for s in g["score_aug"]]) for g in guesses])
    return inf_score + aug_score

def score_full_probmul_3(guesses):
    return score_sum(guesses, getter_full_probmul_3)

def getter_kgmon(guesses):
    inf_score = len(guesses)
    aug_score = np.mean([np.mean(g["score_aug"]) for g in guesses])
    return inf_score - aug_score

def score_kgmon(guesses):
    return score_sum(guesses, getter_kgmon)


selection_algorithms = [
    score_full_probmul_3,
    score_kgmon,
]


class ArcDecoder:
    
    def __init__(self, dataset, n_guesses):
        self.dataset = dataset
        self.n_guesses = n_guesses
        self.decoded_results = {}

    def load_decoded_results(self, store, run_name=""):
        for key in os.listdir(store):
            with bz2.BZ2File(os.path.join(store, key)) as f:
                outputs = pickle.load(f)
            base_key = key.split(".")[0]
            self.decoded_results[base_key] = self.decoded_results.get(base_key, {})
            for i, sample in enumerate(outputs):
                self.decoded_results[base_key][f"{key}{run_name}.out{i}"] = sample

    def run_selection_algo(self, selection_algorithm=score_kgmon):
        return {bk: selection_algorithm({k: g for k, g in v.items()}) for bk, v in self.decoded_results.items()}

    def benchmark_selection_algos(self):
        print("*** Benchmark selection algorithms...")

        labels = {}
        num_tasks_per_puzzle = {}
        num_solved_keys = 0
        num_total_keys = 0

        correct_beam_scores = []

        for basekey, basevalues in self.decoded_results.items():

            mult_key, mult_sub = basekey.split("_")
            num_tasks_per_puzzle[mult_key] = max(num_tasks_per_puzzle.get(mult_key, 0), int(mult_sub) + 1)

            labels[basekey] = correct_solution = self.dataset.replies[basekey][0]

            for subkey, sample in basevalues.items():

                solution = sample["solution"]
                beam_score = sample["beam_score"]
                aug_mean = np.mean(sample["score_aug"])

                if np.shape(correct_solution) != np.shape(solution):
                    corr_str = "bad_xy_size"
                elif np.array_equal(correct_solution, solution):
                    corr_str = "ALL_CORRECT"
                    num_solved_keys += 1
                    correct_beam_scores.append(beam_score)
                else:
                    corr_str = "bad_content"

                output_len = f"{solution.shape[0]}x{solution.shape[1]}"

                if corr_str == "ALL_CORRECT":
                    print(f"{corr_str}:{beam_score:8.5f} - {aug_mean:8.5f} {output_len:5s} [{subkey}]")
                num_total_keys += 1

        print(f" subkeys: {num_solved_keys}/{num_total_keys}")
        print(f" avg correct beam score: {np.mean(correct_beam_scores):8.5f}")
        print(f" max correct beam score: {np.max(correct_beam_scores):8.5f}")

        num_puzzles = len(num_tasks_per_puzzle)

        for selection_algorithm in selection_algorithms:
            name = selection_algorithm.__name__
            selected = self.run_selection_algo(selection_algorithm)
            correct_puzzles = {k for k, v in selected.items() if any(np.array_equal(guess, labels[k]) for guess in v[:self.n_guesses])}
            print(correct_puzzles)
            score = sum(1/num_tasks_per_puzzle[k.split("_")[0]] for k in correct_puzzles)
            print(f" acc: {score:5.1f}/{num_puzzles:3} ('{name}')")

In [ ]:
%%writefile arc_solver.py
# HARDENED_FOR_SUBMISSION: fp16 GradScaler bypass. Turing (T4) lacks native
# bf16, so accelerate takes the fp16 path and unsloth still drives GradScaler.
def _disable_fp16_grad_scaler():
    import torch.amp.grad_scaler as grad_scaler
    grad_scaler.GradScaler._unscale_grads_ = lambda *args, **kwargs: None
    grad_scaler.GradScaler.unscale_ = lambda *args, **kwargs: None
    grad_scaler.GradScaler.step = lambda self, opt, *a, **k: opt.step(*a, **k)
    grad_scaler.GradScaler.update = lambda *args, **kwargs: None


try:
    _disable_fp16_grad_scaler()
    print("GradScaler fp16 unscaling disabled", flush=True)
except Exception as error:
    print(f"GradScaler patch skipped: {error}", flush=True)
from unsloth import FastLanguageModel, UnslothTrainingArguments, UnslothTrainer
from arc_loader import ArcDataset, QwenFormatter
import arc_schedule

# Re-apply after unsloth imports so its patched torch copy cannot win.
try:
    _disable_fp16_grad_scaler()
except Exception:
    pass

import gc
import os
import io
import time
import traceback
import torch
import numpy as np
from tqdm import tqdm
from datasets import Dataset
from collections import defaultdict

from typing import Any, Union
from transformers import DataCollatorForLanguageModeling

import logging
from contextlib import redirect_stdout, redirect_stderr

from peft import get_peft_model_state_dict, set_peft_model_state_dict

import bz2
import pickle

logging.disable(logging.WARNING)

ARC_VOCAB = {
    "0": 0,
    "1": 1,
    "2": 2,
    "3": 3,
    "4": 4,
    "5": 5,
    "6": 6,
    "7": 7,
    "8": 8,
    "9": 9,
    "Ċ": 10,
    "<|im_end|>": 15,
}

ARC_TOKENS = list(ARC_VOCAB.values())
USER_TOKEN_ID = 11
ASSISTANT_TOKEN_ID = 12
PAD_ID = 13
EOS_ID = 15


class UnslothFixedTrainer(UnslothTrainer):

    # Issue https://github.com/unslothai/unsloth/issues/2435

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        """Fixed compute_loss that handles Unsloth's view tensor issue"""
        if self.label_smoother is not None and "labels" in inputs:
            labels = inputs.pop("labels")
        else:
            labels = None
        outputs = model(**inputs)
        if labels is not None:
            unwrapped_model = self.accelerator.unwrap_model(model)
            if hasattr(unwrapped_model, "_get_name") and "unsloth" in unwrapped_model._get_name().lower():
                loss = self.label_smoother(outputs, labels, shift_labels=True)
            else:
                loss = self.label_smoother(outputs, labels)
        else:
            loss = outputs["loss"] if isinstance(outputs, dict) else outputs[0]
        # 🔧 KEY FIX: Clone the loss tensor before in-place operations
        if hasattr(loss, "clone"):
            loss = loss.clone()  # Converts view tensor to independent tensor
        # Now safe for DDP gradient scaling
        if self.accelerator.num_processes > 1:
            loss = loss * self.accelerator.num_processes
        return (loss, outputs) if return_outputs else loss


class QwenDataCollatorForCompletionOnlyLM(DataCollatorForLanguageModeling):

    def torch_call(self, examples: list[Union[list[int], Any, dict[str, Any]]]) -> dict[str, Any]:
        batch = super().torch_call(examples)
        for i in range(len(examples)):
            labels = batch["input_ids"][i].clone()
            user_start_idx = np.where(labels == USER_TOKEN_ID)[0].tolist()
            assistant_start_idx = np.where(labels == ASSISTANT_TOKEN_ID)[0].tolist()
            start_idx = sorted(user_start_idx + assistant_start_idx)
            end_idx = np.where(labels == EOS_ID)[0]
            batch["labels"][i, :] = -100
            for j, (start, end) in enumerate(zip(start_idx, end_idx)):
                assert start < end
                if j % 2 == 1:
                    start += 2
                    end += 1
                    batch["labels"][i, start:end] = labels[start:end]
        return batch


# Minimal performance patch: preserve the baseline beam set and ranking, but transfer
# only the 12 ARC-token NLL values to CPU instead of every Qwen vocabulary logit.
_ARC_TOKEN_ID_CACHE = {}


def _arc_token_ids(device):
    key = str(device)
    token_ids = _ARC_TOKEN_ID_CACHE.get(key)
    if token_ids is None:
        token_ids = torch.tensor(ARC_TOKENS, dtype=torch.long, device=device)
        _ARC_TOKEN_ID_CACHE[key] = token_ids
    return token_ids


def turbo_dfs(model, logits, max_new_tokens, max_score, scores, pos, cache, start_time, end_time) -> dict:

    n = logits.size(0)

    # Algebraically identical to: scores - logits.float().cpu().log_softmax(-1),
    # restricted to the same ARC_TOKENS used by the baseline DFS loop.
    logits_f = logits.float()
    token_ids = _arc_token_ids(logits.device)
    arc_logits = logits_f.index_select(-1, token_ids)
    nll = (
        torch.as_tensor(scores, dtype=torch.float32, device=logits.device).view(n, 1)
        + torch.logsumexp(logits_f, dim=-1, keepdim=True)
        - arc_logits
    ).cpu()

    suffixes = defaultdict(list)

    candidates = dict()

    for i in range(n):
        candidates[i] = []
        for token_idx, t in enumerate(ARC_TOKENS):
            score = nll[i, token_idx].item()
            if score < max_score:
                if t == EOS_ID:
                    suffixes[i].append((score, [t]))
                elif max_new_tokens > 1:
                    candidates[i].append((score, t))

    for i in range(n):
        candidates[i] = sorted(candidates[i], key=lambda x:x[0]) #[:5]
    
    while time.time() - start_time < 540 and time.time() < end_time:

        batch_tokens = []
        batch_scores = []
        num_alive_beams = 0

        for i in range(n):
            if len(candidates[i]) == 0:
                batch_tokens.append(PAD_ID)
                batch_scores.append(1000)
            else:
                score, t = candidates[i].pop(0)
                batch_tokens.append(t)
                batch_scores.append(score)
                num_alive_beams += 1

        if num_alive_beams == 0:
            break

        outputs = model(
            input_ids=torch.tensor(batch_tokens, device=model.device, dtype=torch.long).view(-1, 1),
            position_ids=torch.full((n, 1), pos, device=model.device),
            past_key_values=cache,
            return_dict=True,
            use_cache=True,
        )

        next_suffixes = turbo_dfs(
            model,
            logits=outputs.logits[:, -1],
            max_new_tokens=max_new_tokens-1,
            max_score=max_score,
            scores=batch_scores,
            pos=pos+1,
            cache=outputs.past_key_values,
            start_time=start_time,
            end_time=end_time,
        )

        for batch_id, beams in next_suffixes.items():
            for score, suffix_tokens in beams:
                suffix_tokens.insert(0, batch_tokens[batch_id])
                suffixes[batch_id].append((score, suffix_tokens))

    return suffixes


@torch.no_grad()
def inference_turbo_dfs(model, prefix_tokens, max_new_tokens, max_score, end_time):
    input_ids = torch.tensor(prefix_tokens, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    suffixes = turbo_dfs(
        model,
        logits=outputs.logits[:, -1],
        max_new_tokens=max_new_tokens,
        max_score=max_score,
        scores=[0.0] * input_ids.size(0),
        pos=input_ids.size(1),
        cache=outputs.past_key_values,
        start_time=time.time(),
        end_time=end_time,
    )
    result = []
    for batch_id, beams in suffixes.items():
        sorted_beams = sorted(beams, key=lambda x:x[0])
        result.append((batch_id, sorted_beams))
    return result


@torch.no_grad()
def calc_scores(queries, answers, tokenizer, model):
    batch_query_tokens = []
    batch_answer_tokens = []
    batch_tokens = []
    batch_lengths = []
    for query, answer in zip(queries, answers):
        query_tokens = tokenizer.encode(query)
        answer_tokens = tokenizer.encode(answer)
        tokens = query_tokens + answer_tokens
        batch_query_tokens.append(query_tokens)
        batch_answer_tokens.append(answer_tokens)
        batch_tokens.append(tokens)
        batch_lengths.append(len(tokens))
    max_len = max(batch_lengths)
    padded_tokens = []
    for tokens in batch_tokens:
        padded = tokens + [PAD_ID] * (max_len - len(tokens))
        padded_tokens.append(padded)
    input_ids = torch.tensor(padded_tokens, device=model.device, dtype=torch.long)

    # Keep logits on GPU and gather only the target-token scores. KV cache is not
    # consumed by teacher-forced scoring, so disabling it removes redundant writes.
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=False)
    batch_logits = outputs.logits.float()
    batch_log_norm = torch.logsumexp(batch_logits, dim=-1)
    result = []
    for row_id, (query_tokens, answer_tokens) in enumerate(zip(batch_query_tokens, batch_answer_tokens)):
        query_length = len(query_tokens)
        answer_length = len(answer_tokens)
        positions = torch.arange(
            query_length - 1,
            query_length - 1 + answer_length,
            device=model.device,
        )
        target_tokens = torch.tensor(answer_tokens, device=model.device, dtype=torch.long)
        answer_log_probs = (
            batch_logits[row_id, positions, target_tokens]
            - batch_log_norm[row_id, positions]
        )
        result.append(-answer_log_probs.sum().item())
    return result


def worker(rank, queue, end_time, task_counter=None):

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")
    num_workers = int(os.environ.get("ARC_NUM_WORKERS", "1"))
    total_tasks = int(os.environ.get("ARC_TOTAL_TASKS", "0"))

    peft_params = dict(
        r=256,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "embed_tokens", "lm_head"],
        lora_alpha=32,
        lora_dropout=0.0,
        bias="none",
        use_gradient_checkpointing=False,
        random_state=42,
        use_rslora=True,
        loftq_config=None,
    )

    train_args = dict(
        per_device_eval_batch_size=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        num_train_epochs=1,
        warmup_steps=0,
        warmup_ratio=0.1,
        max_grad_norm=1.0,
        learning_rate=5e-5,
        optim="adamw_torch",
        weight_decay=0.0,
        lr_scheduler_type="cosine",
        seed=42,
        report_to="none",
        save_strategy="no",
        eval_strategy="no",
        logging_strategy="no",
        fp16=False,
        bf16=torch.cuda.is_bf16_supported(),
        # Disable FSDP (use standard DDP)
        fsdp="",
        ddp_find_unused_parameters=False,
        dataloader_num_workers=0,
        gradient_checkpointing=False,
    )

    max_seq_length = 8192

    print(
        f"[Rank {rank}] cuda_devices={torch.cuda.device_count()} "
        f"device0={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'} "
        f"bf16_supported={torch.cuda.is_bf16_supported()} "
        f"train_precision={'bf16' if torch.cuda.is_bf16_supported() else 'fp16'}",
        flush=True,
    )

    model_dir = arc_schedule.find_model_dir()
    if model_dir is None:
        raise FileNotFoundError(
            "attached Qwen3-4B weights not found under /kaggle/input; "
            "check kernel metadata model_sources"
        )
    print(f"[Rank {rank}] model dir: {model_dir}", flush=True)

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_dir,
        full_finetuning=False,
        load_in_4bit=False,
        local_files_only=True,
        use_gradient_checkpointing=False,
        max_seq_length=max_seq_length,
    )

    model = FastLanguageModel.get_peft_model(model, **peft_params)

    target_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    if not torch.cuda.is_bf16_supported():
        model = model.to(target_dtype)
    for name, param in model.named_parameters():
        if param.dtype == torch.float32:
            param.data = param.data.to(target_dtype)

    default_weights = get_peft_model_state_dict(model, adapter_name="default")
    default_weights = {k: v.clone().detach() for k, v in default_weights.items()}

    collator = QwenDataCollatorForCompletionOnlyLM(
        tokenizer=tokenizer,
        mlm=False,
    )

    formatter = QwenFormatter(tokenizer=tokenizer)

    max_new_tokens = formatter.max_new_tokens()

    max_score = -np.log(0.2)

    test_path = os.environ.get("ARC_CHALLENGE_PATH")
    if not test_path:
        suffix = "test" if rerun_mode else "evaluation"
        test_path = f"/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_{suffix}_challenges.json"
    print(f"[Rank {rank}] challenge file: {test_path}", flush=True)

    arc_test_set = ArcDataset.from_file(test_path)

    dir_outputs = "/kaggle/inference_outputs"
    os.makedirs(dir_outputs, exist_ok=True)

    while not queue.empty():

        if time.time() > end_time:
            print(f"[Rank {rank}] stop!")
            break
        key = queue.get()
        if key is None:
            break
        # HARDENED_FOR_SUBMISSION: isolate each task so one failure cannot
        # end the worker and forfeit every remaining task in the 12h budget.
        try:

            # SCHEDULED_FOR_COVERAGE: the fair share of the remaining wall clock
            # across the tasks still in the queue. The baseline used a constant
            # 1200s cap, which is 1.7x the 710s share that 240 hidden tasks over
            # 4 GPUs actually allow, and it starved the tail of the queue.
            tasks_left = total_tasks if task_counter is None else max(1, task_counter.value)
            if task_counter is not None:
                task_counter.value = max(0, task_counter.value - 1)
            budget = arc_schedule.task_budget(
                end_time, tasks_left=tasks_left, n_workers=num_workers
            )
            task_end = time.time() + budget
            print(
                f"[Rank {rank}] task {key}: {tasks_left} left, "
                f"budget {budget:.0f}s, deadline in {budget:.0f}s",
                flush=True,
            )

            start_time = time.time()
        
            torch.cuda.reset_peak_memory_stats()

            load_result = set_peft_model_state_dict(
                model,
                default_weights.copy(),
                adapter_name="default",
            )

            model = FastLanguageModel.for_training(model)

            puzzle_ds = arc_test_set.change_keys([key])

            train_ds = puzzle_ds.augment(n=16, shfl_keys=True, seed=1)
            train_ds = train_ds.cut_to_len(formatter=formatter, name="text", max_len=max_seq_length)

            with io.StringIO() as buf, redirect_stdout(buf), redirect_stderr(buf):
            
                trainer = UnslothFixedTrainer(
                    model=model,
                    tokenizer=tokenizer,
                    data_collator=collator,
                    train_dataset=Dataset.from_list(train_ds.as_list(formatter)),
                    dataset_text_field="text",
                    max_seq_length=max_seq_length,
                    args=UnslothTrainingArguments(**train_args),
                )

                stats = trainer.train()

                model = trainer.accelerator.unwrap_model(model, keep_fp32_wrapper=False)

                del trainer

            model = FastLanguageModel.for_inference(model)
        
            gc.collect()
            torch.cuda.empty_cache()
            
            train_peak_mb = torch.cuda.max_memory_allocated() // 1024**2
            print(f"[Rank {rank}] allocated {train_peak_mb}MB for training")

            torch.cuda.reset_peak_memory_stats()
        
            print(f"[Rank {rank}] training stats for puzzle {key}: {stats}")

            puzzle_ds_multi = puzzle_ds.split_multi_replies()

            eval_ds = puzzle_ds_multi.augment(n=2, seed=2)
            eval_ds = eval_ds.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)

            test_id_to_subkeys = defaultdict(list)
            for subkey in sorted(eval_ds.keys):
                test_id = subkey.split(".")[0].split("_")[1]
                test_id_to_subkeys[test_id].append(subkey)

            # SCHEDULED_FOR_COVERAGE: same offset grouping as the baseline
            # (0/4, 2/6, 8/12, 10/14 -> two dihedral views per jointly decoded
            # batch of 4), but derived from the real key order and asserted to
            # cover all 16 augmented views exactly once.
            batches = []
            for test_id, subkeys in test_id_to_subkeys.items():
                for batch in arc_schedule.plan_decode_batches(subkeys):
                    batches.append((test_id, batch))
            n_batches_per_test = max(
                [1] + [
                    sum(1 for tid, _ in batches if tid == test_id)
                    for test_id, _ in batches
                ]
            )
            dfs_cap = arc_schedule.dfs_time_cap(budget, n_batches=n_batches_per_test)
            print(
                f"[Rank {rank}] {len(batches)} decode batches, "
                f"{dfs_cap:.0f}s per batch for {key}",
                flush=True,
            )

            with torch.inference_mode():
                
                known_scores = {}
                # INSTRUMENT: counters for the per-task failure table.
                batches_decoded = 0
                candidate_count = 0

                for _test_id, subkeys in batches:

                    spend_time = time.time() - start_time
                    if time.time() >= task_end or budget <= 0:
                        print(
                            f"[Rank {rank}] task budget spent after {spend_time:.1f}s "
                            f"for puzzle {key}"
                        )
                        break

                    print(f"[Rank {rank}] decoding {subkeys}")

                    tokens = []
                    for subkey in subkeys:
                        data = eval_ds.get(subkey, formatter)
                        tokens.append(tokenizer.encode(data["input"]))

                    batch_deadline = min(task_end, time.time() + dfs_cap)
                    dfs_result = inference_turbo_dfs(
                        model, tokens, max_new_tokens, max_score, batch_deadline
                    )
                    batches_decoded += 1

                    for subkey_id, scored_beams in dfs_result:

                        subkey = subkeys[subkey_id]
                        bk = subkey.split(".")[0]
                        decoded_result = []

                        for beam_score, tokens in scored_beams:

                            array = formatter.convert_tokens_to_array(tokens)
                            if array is None:
                                continue

                            solution = puzzle_ds_multi.invert_mod(array, subkey, inv_perm=True)

                            grid_id = (bk, tuple(map(tuple, solution)))

                            if grid_id in known_scores:
                                augmented_scores = known_scores[grid_id]
                            else:
                                print(f"[Rank {rank}] scoring {subkey} #{len(decoded_result)}")
                                aug_dataset = ArcDataset(
                                    keys=[bk],
                                    queries={bk: puzzle_ds_multi.queries.get(bk)},
                                    replies={bk: [solution.tolist()]},
                                )
                                aug_dataset = aug_dataset.augment(seed=hash(bk) % 1024**2)
                                aug_dataset = aug_dataset.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)
                                aug_queries = []
                                aug_answers = []
                                for augmented_sample in aug_dataset.as_list(formatter):
                                    aug_queries.append(augmented_sample["input"])
                                    aug_answers.append(augmented_sample["reply"])
                                augmented_scores1 = calc_scores(aug_queries[:4], aug_answers[:4], tokenizer, model)
                                augmented_scores2 = calc_scores(aug_queries[4:], aug_answers[4:], tokenizer, model)
                                augmented_scores = augmented_scores1 + augmented_scores2
                                known_scores[grid_id] = augmented_scores
                        
                            decoded_result.append({
                                "beam_score": beam_score,
                                "score_aug": augmented_scores,
                                "solution": solution,
                            })

                        candidate_count += len(decoded_result)
                        if len(decoded_result):
                            with bz2.BZ2File(os.path.join(dir_outputs, subkey), "w") as f:
                                pickle.dump(decoded_result, f)

            infer_peak_mb = torch.cuda.max_memory_allocated() // 1024**2
            print(f"[Rank {rank}] allocated {infer_peak_mb}MB for inference")
        
            spend_time = time.time() - start_time
            print(f"[Rank {rank}] finished {key} in {spend_time:.1f}s")

            # INSTRUMENT: one row per task so coverage, truncation and cost can be
            # analysed without re-running the 12h notebook.
            try:
                metrics_path = f"/kaggle/working/task_metrics_rank{rank}.csv"
                header = (
                    "key,budget_s,batches_planned,batches_decoded,candidates,"
                    "runtime_s,train_peak_mb,infer_peak_mb\n"
                )
                exists = os.path.exists(metrics_path)
                with open(metrics_path, "a") as handle:
                    if not exists:
                        handle.write(header)
                    handle.write(
                        f"{key},{budget:.1f},{len(batches)},{batches_decoded},"
                        f"{candidate_count},{spend_time:.1f},{train_peak_mb},"
                        f"{infer_peak_mb}\n"
                    )
            except Exception:
                traceback.print_exc()
        except Exception:
            traceback.print_exc()
            print(f"[Rank {rank}] task {key} failed; continuing", flush=True)
            continue



In [ ]:
%%writefile starter.py
import os
import time
import json
import torch
import argparse
import torch.multiprocessing as mp

import arc_schedule

COMPETITION_DIRS = [
    "/kaggle/input/competitions/arc-prize-2026-arc-agi-2",
    "/kaggle/input/arc-prize-2026-arc-agi-2",
]
# SCHEDULED_FOR_COVERAGE: the previous revision queued only these four
# hard-coded evaluation task ids unless KAGGLE_IS_COMPETITION_RERUN was set, so a
# scored rerun without that variable would emit 236 placeholders. Smoke runs are
# now opt-in through ARC_SMOKE_KEYS.
INFERENCE_OUTPUTS = "/kaggle/inference_outputs"


def resolve_challenge_file():
    """HARDENED_FOR_SUBMISSION: never assume a single mount layout."""
    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")
    wanted = "arc-agi_test_challenges.json" if rerun_mode else "arc-agi_evaluation_challenges.json"
    override = os.environ.get("ARC_CHALLENGE_PATH")
    if override and os.path.exists(override):
        return override
    for directory in COMPETITION_DIRS:
        candidate = os.path.join(directory, wanted)
        if os.path.exists(candidate):
            return candidate
    for root, _dirs, names in os.walk("/kaggle/input"):
        for name in names:
            if name == wanted:
                return os.path.join(root, name)
    raise FileNotFoundError(
        f"could not locate {wanted}; /kaggle/input contains "
        + json.dumps(sorted(os.listdir("/kaggle/input")))
    )


def local_worker(rank, queue, end_time, task_counter, num_workers):

    os.environ["CUDA_VISIBLE_DEVICES"] = str(rank)
    os.environ["ARC_NUM_WORKERS"] = str(num_workers)

    torch.set_default_device("cpu")

    # Fix Unsloth patching issue
    if rank > 0:
        while not os.path.exists(f"/kaggle/worker{rank-1}"):
            time.sleep(5)

    from arc_solver import worker

    with open(f"/kaggle/worker{rank}", "w") as f:
        f.write("Ok")

    print(f"[Rank {rank}] start!", flush=True)

    worker(rank, queue, end_time, task_counter)

    print(f"[Rank {rank}] done!", flush=True)


if __name__ == "__main__":

    parser = argparse.ArgumentParser()
    parser.add_argument("--end-time", type=float, default=0.0)
    args = parser.parse_args()
    if args.end_time <= 0.0:
        args.end_time = time.time() + 12 * 3600 - 600

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    test_path = resolve_challenge_file()
    print(f"starter.py challenge file: {test_path} (rerun_mode={bool(rerun_mode)})", flush=True)
    os.environ["ARC_CHALLENGE_PATH"] = test_path

    with open(test_path, "r") as f:
        data = json.load(f)

    os.makedirs(INFERENCE_OUTPUTS, exist_ok=True)

    manager = mp.Manager()
    queue = manager.Queue()

    smoke_keys = os.environ.get("ARC_SMOKE_KEYS", "").replace(",", " ").split() or None
    keys = arc_schedule.queue_keys(data, smoke_keys=smoke_keys)
    print(f"starter.py queueing {len(keys)} tasks: {keys}", flush=True)
    for key in keys:
        queue.put(key)
    num_workers = max(1, torch.cuda.device_count())
    print(f"starter.py detected {num_workers} CUDA devices", flush=True)
    for _ in range(num_workers):
        queue.put(None)

    task_counter = manager.Value("i", len(keys))
    os.environ["ARC_TOTAL_TASKS"] = str(len(keys))
    print(
        "starter.py coverage plan: "
        + json.dumps(
            arc_schedule.coverage_report(
                args.end_time, time.time(), len(keys), num_workers
            )
        ),
        flush=True,
    )

    mp.spawn(
        local_worker,
        args=(queue, args.end_time, task_counter, num_workers),
        nprocs=num_workers,
    )


In [ ]:
%%writefile arc_schedule.py
"""Time budgeting and augmentation-batch planning for the ARC-AGI-2 TTT notebook.

Standard library only, so the exact code that runs on Kaggle can be imported and
unit-tested on CPU.  The GPU cell (`arc_solver.py`) imports this module; the
notebook ships it verbatim via ``%%writefile``.

Motivation (measured arithmetic, see NOTES.md): the 33.89 baseline gives every
task a hard 1200 s wall-clock allowance while `turbo_dfs` is allowed to burn
540 s per augmentation batch against the *global* deadline.  With 240 hidden
tasks, 4 GPUs and a 12 h budget the fair share is ~1190 s per task, so a single
task can overrun its share by ~36% and the FIFO queue starves the tail tasks,
which then emit the `[[0]]` placeholder.  Every task must get an attempt, and
every augmentation view must get an equal share of the task budget.
"""

import time

MAX_TASK_BUDGET = 1500.0
HARD_DFS_CAP = 540.0
MIN_DFS_CAP = 20.0
BATCHES_PER_TEST = 4


def task_budget(end_time, now=None, tasks_left=1, n_workers=1,
                max_budget=MAX_TASK_BUDGET):
    """Seconds this worker may spend on the task it is about to start.

    With the verified 12 h budget (42600 s after the write buffer), 240 hidden
    tasks and 4 GPUs the fair share is 710 s per task, while the baseline's hard
    cap is 1200 s and its worst case is 1620 s.

    ``tasks_left`` counts every task still unclaimed across all workers, so the
    budget self-corrects as the queue drains instead of assuming a uniform cost.
    There is deliberately no minimum allowance: a floor larger than the fair share
    reintroduces exactly the tail starvation this replaces. A queue that cannot be
    covered in the time available is reported by `coverage_report` rather than
    hidden by a floor. The result never exceeds the time left before the deadline.
    """
    now = time.time() if now is None else now
    remaining = end_time - now
    if remaining <= 0:
        return 0.0
    fair_share = remaining * max(1, n_workers) / max(1, tasks_left)
    return min(max_budget, fair_share, remaining)


def dfs_time_cap(budget, n_batches=BATCHES_PER_TEST,
                 hard_cap=HARD_DFS_CAP, floor=MIN_DFS_CAP):
    """Per-batch decode allowance so all augmentation views are covered evenly.

    The baseline let each batch consume up to 540 s and only re-checked the task
    cap *between* batches, so a task could burn 1620 s and, inside a 1200 s
    allowance, decode only 3 of its 4 batches (12 of 16 inference views).
    Splitting the task budget evenly over all 4 batches keeps all 16 views --
    and therefore the full rescoring vote -- inside the smaller fair share.
    """
    if n_batches <= 0:
        return hard_cap
    return max(floor, min(hard_cap, budget / float(n_batches)))


def plan_decode_batches(subkeys):
    """Group the sorted augmented subkeys of one test output into decode batches.

    Reproduces the baseline grouping (offset pairs 0/4, 2/6, 8/12, 10/14, two
    subkeys each -> 4 jointly decoded batches of 4) and asserts that all views
    are scheduled exactly once.
    """
    keys = list(subkeys)
    if not keys:
        return []
    if len(keys) < 2 * BATCHES_PER_TEST:
        return [keys]
    offset_pairs = ((0, 4), (2, 6), (8, 12), (10, 14))
    batches = []
    for first, second in offset_pairs:
        batch = keys[first:first + 2] + keys[second:second + 2]
        batches.append(batch)
    scheduled = [key for batch in batches for key in batch]
    if sorted(scheduled) != sorted(keys):
        raise AssertionError("decode batches do not cover every augmented view once")
    return batches


def coverage_report(end_time, now, n_tasks, n_workers):
    """Human-readable projection of queue coverage, for the notebook log."""
    budget = task_budget(end_time, now=now, tasks_left=n_tasks, n_workers=n_workers)
    cap = dfs_time_cap(budget)
    needed = n_tasks * budget / max(1, n_workers)
    return {
        "tasks": n_tasks,
        "workers": n_workers,
        "wall_clock_left_s": round(end_time - now, 1),
        "per_task_budget_s": round(budget, 1),
        "dfs_cap_per_batch_s": round(cap, 1),
        "projected_total_s": round(needed, 1),
        "fits_budget": needed <= (end_time - now) * max(1, n_workers),
        "n_batches": BATCHES_PER_TEST,
    }


def queue_keys(data, smoke_keys=None):
    """Task keys to attempt.

    The baseline queued only four hard-coded evaluation task ids unless
    `KAGGLE_IS_COMPETITION_RERUN` was set, which would forfeit ~98% of the
    hidden set if that variable is ever absent during a scored rerun.  All keys
    are always queued; a smoke run has to be requested explicitly.
    """
    keys = sorted(data.keys())
    if smoke_keys:
        selected = [k for k in smoke_keys if k in data]
        if selected:
            return selected
    return keys


def find_model_dir(root="/kaggle/input", marker="qwen3_4b_grids15_sft139"):
    """Locate the attached model directory instead of trusting one hard-coded path.

    Kaggle mounts attached model datasets under versioned prefixes, so a single
    hard-coded path is a single point of total failure (0% submission).
    """
    import os

    direct = os.path.join(root, "models", "sorokin", marker, "transformers", "bfloat16", "1")
    if os.path.isdir(direct):
        return direct
    for dirpath, _dirnames, _filenames in os.walk(root):
        if os.path.basename(dirpath) == marker or marker in dirpath.split(os.sep):
            if os.path.isfile(os.path.join(dirpath, "config.json")):
                return dirpath
    return None


In [ ]:
import os, sys, time, subprocess, traceback

end_time = time.time() + 12 * 3600 - 600
env = os.environ.copy()
env["UNSLOTH_DISABLE_STATISTICS"] = "1"
env["TRITON_PTXAS_PATH"] = "/usr/local/cuda/bin/ptxas"
env["OMP_NUM_THREADS"] = "12"
env["PYTHONUNBUFFERED"] = "1"
# SCHEDULED_FOR_COVERAGE: arc_solver seeds augmentation with hash(bk), which is
# randomised per process without this, so the rescoring sets were not comparable
# across workers.
env["PYTHONHASHSEED"] = "0"
cmd = [sys.executable, "starter.py", "--end-time", f"{end_time:.2f}"]
print(f"Launching starter: {' '.join(cmd)}", flush=True)
try:
    res = subprocess.run(cmd, env=env)
    print(f"starter.py finished with returncode {res.returncode}", flush=True)
except BaseException:
    # HARDENED_FOR_SUBMISSION: never let the solver abort the submission cell.
    traceback.print_exc()
    print("starter.py raised; continuing to the submission cell", flush=True)


In [ ]:
import os
import json
import traceback
from arc_loader import ArcDataset
from arc_decoder import ArcDecoder

OUTPUT_PATH = "/kaggle/working/submission.json"
INFERENCE_OUTPUTS = "/kaggle/inference_outputs"
COMPETITION_DIRS = [
    "/kaggle/input/competitions/arc-prize-2026-arc-agi-2",
    "/kaggle/input/arc-prize-2026-arc-agi-2",
]


def resolve_challenge_file():
    """HARDENED_FOR_SUBMISSION: never assume a single mount layout."""
    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")
    wanted = "arc-agi_test_challenges.json" if rerun_mode else "arc-agi_evaluation_challenges.json"
    override = os.environ.get("ARC_CHALLENGE_PATH")
    if override and os.path.exists(override):
        return override
    for directory in COMPETITION_DIRS:
        candidate = os.path.join(directory, wanted)
        if os.path.exists(candidate):
            return candidate
    for root, _dirs, names in os.walk("/kaggle/input"):
        for name in names:
            if name == wanted:
                return os.path.join(root, name)
    raise FileNotFoundError(
        f"could not locate {wanted}; /kaggle/input contains "
        + json.dumps(sorted(os.listdir("/kaggle/input")))
    )


def resolve_solutions_file():
    """HARDENED_FOR_SUBMISSION: locate the public evaluation answers."""
    wanted = "arc-agi_evaluation_solutions.json"
    for directory in COMPETITION_DIRS:
        candidate = os.path.join(directory, wanted)
        if os.path.exists(candidate):
            return candidate
    for root, _dirs, names in os.walk("/kaggle/input"):
        for name in names:
            if name == wanted:
                return os.path.join(root, name)
    raise FileNotFoundError(f"could not locate {wanted}")


test_path = resolve_challenge_file()
print(f"Loading challenge test set from: {test_path}", flush=True)
data = ArcDataset.from_file(test_path)

decoder = ArcDecoder(data.split_multi_replies(), n_guesses=2)

os.makedirs(INFERENCE_OUTPUTS, exist_ok=True)
try:
    decoder.load_decoded_results(INFERENCE_OUTPUTS)
    results = decoder.run_selection_algo()
    print(f"neural candidate sets for {len(results)} tasks", flush=True)
except Exception:
    # HARDENED_FOR_SUBMISSION: a dead solver must still yield a schema-valid file.
    traceback.print_exc()
    results = {}
    print("no usable neural results; emitting placeholder predictions", flush=True)

try:
    submission = data.get_submission(results)
except Exception:
    traceback.print_exc()
    results = {}
    print("candidate fill failed; emitting placeholder predictions", flush=True)
    submission = data.get_submission(results)

with open(OUTPUT_PATH, "w") as f:
    json.dump(submission, f)

with open(OUTPUT_PATH, "r") as f:
    reloaded = json.load(f)

assert list(reloaded) == list(data.keys), "submission task IDs or order do not match the challenge"
placeholder = 0
solved = 0
for key, entries in reloaded.items():
    expected = len(data.queries[key]["test"])
    assert len(entries) == expected, f"{key}: expected {expected} outputs, got {len(entries)}"
    for entry in entries:
        assert set(entry) == {"attempt_1", "attempt_2"}, f"{key}: bad attempt keys"
        for attempt in ("attempt_1", "attempt_2"):
            grid = entry[attempt]
            assert isinstance(grid, list) and grid, f"{key}: empty {attempt}"
            width = len(grid[0])
            assert all(len(row) == width for row in grid), f"{key}: ragged {attempt}"
            assert 0 < len(grid) <= 30 and 0 < width <= 30, f"{key}: {attempt} exceeds 30x30"
            assert all(
                isinstance(value, int) and 0 <= value <= 9 for row in grid for value in row
            ), f"{key}: value outside 0-9 in {attempt}"
        if entry["attempt_1"] == [[0]]:
            placeholder += 1
        else:
            solved += 1

print(
    f"Wrote and validated {OUTPUT_PATH}: {len(reloaded)} tasks, "
    f"{solved} predicted test items, {placeholder} placeholders",
    flush=True,
)

if not os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
    try:
        solutions = resolve_solutions_file()
        data.load_replies(solutions)
        decoder.dataset = data.split_multi_replies()
        print(f"Evaluation solutions: {solutions}", flush=True)
        print("*** Reload score:", data.validate_submission(reloaded), flush=True)
    except Exception:
        traceback.print_exc()
        print("scoring against the evaluation solutions was not possible", flush=True)

print("DONE_OK", flush=True)
